# MAP Scoring using the Generative Large Language Model (GLLM) Approach


This file serves to create the MAP scores based on the GLLM approach based on 10-K filings of the U.S. S&P 500 firms with filing year 2013 to 2023.
 

<div class='alert-warning'>
Libraries
</div>
First, we Import all necessary libraries. These inlcude 'os' to set and handle working directories, 'pandas' and 'numpy' for data handling and calculations, 'pickle' to load and save the prepared data as memory efficient pickle files, 'lseg.data' to retrieve further information from the LSEG Datastram API, and 'plotnine' to create plots/figures. 

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import pickle
import lseg.data as rd
import plotnine
from scipy.stats.mstats import winsorize
# package for plot scales
from mizani.formatters import comma_format # (thousands seperator format)


<div class='alert-warning'>
Set the working directory
</div>

In [ ]:
# Set working directory 
os.chdir('../../../../data')

<div class='alert-warning'>
Load the data shards from the parallelization of the GLLM approach 
</div>

In [ ]:
#Load the dataset shards of the zero-shot GLLM approach
#Corpus_df_HTML_cleaned_GLLM_final_shard_0.pkl
#Corpus_df_HTML_cleaned_GLLM_final_shard_1.pkl
#Corpus_df_HTML_cleaned_GLLM_final_shard_2.pkl
#Corpus_df_HTML_cleaned_GLLM_final_shard_3.pkl

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_shard_0.pkl','rb') as path_name:
    shard_0 = pickle.load(path_name)

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_shard_1.pkl','rb') as path_name:
    shard_1 = pickle.load(path_name)

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_shard_2.pkl','rb') as path_name:
    shard_2 = pickle.load(path_name)

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_shard_3.pkl','rb') as path_name:
    shard_3 = pickle.load(path_name)

#Combine the shards into one dataframe
corpus_df = pd.concat([shard_0, shard_1, shard_2, shard_3], ignore_index=True)

del shard_0, shard_1, shard_2, shard_3

#Load the dataset shards of the fine-tuning GLLM approach
#Corpus_df_HTML_cleaned_GLLM_final_FT_shard_0.pkl
#Corpus_df_HTML_cleaned_GLLM_final_FT_shard_1.pkl
#Corpus_df_HTML_cleaned_GLLM_final_FT_shard_2.pkl
#Corpus_df_HTML_cleaned_GLLM_final_FT_shard_3.pkl

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_FT_shard_0.pkl','rb') as path_name:
    shard_0 = pickle.load(path_name)

shard_0 = shard_0[['filing_key', 'results', 'prompting_time_minutes']]

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_FT_shard_1.pkl','rb') as path_name:
    shard_1 = pickle.load(path_name)

shard_1 = shard_1[['filing_key', 'results', 'prompting_time_minutes']]

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_FT_shard_2.pkl','rb') as path_name:
    shard_2 = pickle.load(path_name)

shard_2 = shard_2[['filing_key', 'results', 'prompting_time_minutes']]

with open('GLLM/Final_Inference_results/processed/Corpus_df_HTML_cleaned_GLLM_final_FT_shard_3.pkl','rb') as path_name:
    shard_3 = pickle.load(path_name)

shard_3 = shard_3[['filing_key', 'results', 'prompting_time_minutes']]

#Combine the shards into one dataframe
corpus_df_FT = pd.concat([shard_0, shard_1, shard_2, shard_3], ignore_index=True)

del shard_0, shard_1, shard_2, shard_3

corpus_df = corpus_df.merge(corpus_df_FT, on='filing_key', how='left', suffixes=('', '_FT'))

del corpus_df_FT

# Save the corpus with all classification DataFrames included
corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')

In [ ]:
# Quickly compare the prompting times of both GLLM approaches
print('Zero-shot GLLM prompting time (hours): ', corpus_df['prompting_time_minutes'].sum() / 60)
print('Fine-tuning GLLM prompting time (hours): ', corpus_df['prompting_time_minutes_FT'].sum() / 60)

## MAP Dimension Scoring

<div class='alert-info'>
Step 0: Retrieve Industry information at the company-year level
</div>

As we also want to analyze MAP score differences at the industry level we need to receive the industry information. We will use the North American Industry Classification System (NAICS) and get them via the previously retrieved RIC codes ('Instrument Identifier') from the Refinitiv datastream API. 

In [ ]:
#The RIC codes are stored in the 'SP_500_CIK_final' pickle file. So load the file into a dataframe.
with open('SP_500_CIK_final.pkl','rb') as path_name:
    CIK_df = pickle.load(path_name)

#Filter the CIK dataframe such that only matching firms are left and drop duplicates
CIK_df = CIK_df[CIK_df['CIK Number'].isin(corpus_df['CIK'].unique())][['Instrument','CIK Number']].drop_duplicates(subset=['CIK Number'])

#Rename the column for the CIK number in order to have a matching name for the merge-key 
CIK_df.columns = ['Instrument', 'CIK']

#Merge the company identifier (RIC code) to the corpus dataframe
corpus_df = pd.merge(corpus_df, CIK_df, on = 'CIK', how = 'left')

#Define the 'universe' of companies to get data from
universe = corpus_df['Instrument'].unique().dropna()

#Open a new Refinitiv session (make sure that the desktop App is open)
rd.open_session()

#Reqeust the industry data from the API
Industry_df = rd.get_data(universe, ['TR.CIKNUMBER', 'TR.NAICSSector', 'TR.NAICSSubsector', 'TR.NAICSIndustryGroup'])

#Close the Refinitiv session
rd.close_session()

#Filter the output for the CIK codes (or RIC Indentifier) included in the corpus dataframe 
Industry_df = Industry_df[Industry_df['CIK Number'].isin(corpus_df['CIK'])]

#Rename the column for the CIK number (s.o.)
Industry_df.columns = ['Instrument', 'CIK', 'NAICS Sector Name',
       'NAICS Subsector Name', 'NAICS Industry Group Name']

#Merge the industry information to the corpus dataframe
corpus_df = pd.merge(corpus_df, Industry_df[['CIK', 'NAICS Sector Name',
       'NAICS Subsector Name', 'NAICS Industry Group Name']], on = 'CIK', how = 'left')

del Industry_df, universe, CIK_df, path_name

# Next, we perform some data cleaning and filtering steps to prepare the corpus dataframe for further analysis. 

#First, we remove all filings that are in industries with less than 50 filings
corpus_df = corpus_df.dropna(subset=['NAICS Sector Name']).groupby('NAICS Sector Name').filter(lambda x: len(x) > 49)

#Second, we will drop duplicates in the dataframe
corpus_df = corpus_df.drop_duplicates(subset=['filing_key']).reset_index(drop=True)

#Second, rename some industry names to make them shorter
corpus_df.loc[corpus_df['NAICS Sector Name'] == 'Administrative and Support and Waste Management and Remediation Services','NAICS Sector Name'] = 'Administrative and Support Services'
corpus_df.loc[corpus_df['NAICS Sector Name'] == 'Mining, Quarrying, and Oil and Gas Extraction','NAICS Sector Name'] = 'Natural Resource Extraction'
corpus_df.loc[corpus_df['NAICS Sector Name'] == 'Professional, Scientific, and Technical Services','NAICS Sector Name'] = 'Professional and Technical Services'

#Third, save the final data as pickle format
corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')

<div class='alert-info'>
Step 1: Retrieve the Explicit and Implicit MAP-related sentences per filing for each GLLM approach 
</div>

To extract the MAP classification information from the GLLM JSON outputs, we create for each filing and dimension a dictionary where the related sentences and classifications are stored.

In [ ]:
# Define the columns to be retained in the final DataFrames
df_columns = [
    'filing_text',
    'LLM_Confidence_Score',
    'LLM_Dimension',
    'LLM_Explicit_MAP_referral',
    'LLM_Implicit_MAP_referral'
]

# Define all dimensions and the output columns
dimension_map = {
    'Cost': ('Cost_df', 'Cost_FT_df'),
    'Financing / Investment': ('Investment_df', 'Investment_FT_df'),
    'Operations': ('Operations_df', 'Operations_FT_df'),
    'Performance / Internal Reporting': ('Performance_df', 'Performance_FT_df'),
    'Risk / Internal Control': ('Risk_df', 'Risk_FT_df'),
    'Strategy': ('Strategy_df', 'Strategy_FT_df'),
    'Pricing & Revenue Management': ('Pricing_df', 'Pricing_FT_df'),
    'Budgeting / Planning': ('Budget_df', 'Budget_FT_df'),
}

# Create container columns holding empty lists for each dimension
for dim, (col, col_ft) in dimension_map.items():
    corpus_df[col] = [[] for _ in range(len(corpus_df))]
    corpus_df[col_ft] = [[] for _ in range(len(corpus_df))]

# Process all filings
for i in range(len(corpus_df)):

    if i % 50 == 0:
        print(f'Processing filing {i} of {len(corpus_df)}')

    # Build classification tables
    classification_df = pd.concat([
        pd.DataFrame(corpus_df.at[i, 'filing_text'], columns=['filing_text']),
        pd.json_normalize(corpus_df.at[i, 'results'])
    ], axis=1)

    classification_FT_df = pd.concat([
        pd.DataFrame(corpus_df.at[i, 'filing_text'], columns=['filing_text']),
        pd.json_normalize(corpus_df.at[i, 'results_FT'])
    ], axis=1)

    # convert both tables to list of dicts
    rows = classification_df.to_dict('records')
    rows_ft = classification_FT_df.to_dict('records')

    # Process non-FT rows
    for row in rows:
        dim = row.get('LLM_Dimension')
        if dim:
            for key, (col, _) in dimension_map.items():
                if key in dim:
                    corpus_df.at[i, col].append({k: row[k] for k in df_columns})
    
    # Process FT rows
    for row in rows_ft:
        dim_ft = row.get('LLM_Dimension')
        if dim_ft:
            for key, (_, col_ft) in dimension_map.items():
                if key in dim_ft:
                    corpus_df.at[i, col_ft].append({k: row[k] for k in df_columns})


# FINAL STEP: convert all lists → DataFrames 
for idx in range(len(corpus_df)):
    for dim, (col, col_ft) in dimension_map.items():

        corpus_df.at[idx, col] = (
            pd.DataFrame(corpus_df.at[idx, col], columns=df_columns)
            if corpus_df.at[idx, col] else pd.DataFrame(columns=df_columns)
        )

        corpus_df.at[idx, col_ft] = (
            pd.DataFrame(corpus_df.at[idx, col_ft], columns=df_columns)
            if corpus_df.at[idx, col_ft] else pd.DataFrame(columns=df_columns)
        )

del df_columns, dimension_map, dim, col, col_ft, idx, row, rows, rows_ft, classification_df, classification_FT_df

# Save the corpus with all classification DataFrames included
corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')


<div class='alert-info'>
Step 2: Count the number of Explicit and Implicit MAP-related sentences per dimension per filing
</div>

We loop through each filing and count the number of explicit and implicit sentences for each MAP dimension per filing.

In [ ]:
dimension_map = {
    'Cost': ('Cost_df', 'Cost_FT_df'),
    'Financing_Investment': ('Investment_df', 'Investment_FT_df'),
    'Operations': ('Operations_df', 'Operations_FT_df'),
    'Performance_Internal_Reporting': ('Performance_df', 'Performance_FT_df'),
    'Risk_Internal_Control': ('Risk_df', 'Risk_FT_df'),
    'Strategy': ('Strategy_df', 'Strategy_FT_df'),
    'Pricing_Revenue_Management': ('Pricing_df', 'Pricing_FT_df'),
    'Budgeting_Planning': ('Budget_df', 'Budget_FT_df'),
}

for i in range(len(corpus_df)):
    if i % 50 == 0:
        print('Processing filing ', i, ' of ', len(corpus_df))
    for dim, (col, col_ft) in dimension_map.items():
        # Count number of 'Yes' explicit and implicit sentences for non-FT
        df = corpus_df.at[i, col]
        corpus_df.at[i, f'{dim}_explicit_count'] = df['LLM_Explicit_MAP_referral'].eq('Yes').sum()
        corpus_df.at[i, f'{dim}_implicit_count'] = df['LLM_Implicit_MAP_referral'].eq('Yes').sum()
        # Count number of 'Yes' explicit and implicit sentences for FT
        df_ft = corpus_df.at[i, col_ft]
        corpus_df.at[i, f'{dim}_explicit_count_FT'] = df_ft['LLM_Explicit_MAP_referral'].eq('Yes').sum()
        corpus_df.at[i, f'{dim}_implicit_count_FT'] = df_ft['LLM_Implicit_MAP_referral'].eq('Yes').sum()

del dim, col, col_ft, i, df, df_ft, dimension_map

# Save the corpus_df after counting explicit and implicit sentences
corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')

The following code gives a short overview of the raw count variables.

In [ ]:
count_columns = ['Cost_explicit_count', 'Cost_implicit_count', 'Cost_explicit_count_FT', 'Cost_implicit_count_FT',
                    'Financing_Investment_explicit_count', 'Financing_Investment_implicit_count', 'Financing_Investment_explicit_count_FT', 'Financing_Investment_implicit_count_FT',
                    'Operations_explicit_count', 'Operations_implicit_count', 'Operations_explicit_count_FT', 'Operations_implicit_count_FT',
                    'Performance_Internal_Reporting_explicit_count', 'Performance_Internal_Reporting_implicit_count', 'Performance_Internal_Reporting_explicit_count_FT', 'Performance_Internal_Reporting_implicit_count_FT',
                    'Risk_Internal_Control_explicit_count', 'Risk_Internal_Control_implicit_count', 'Risk_Internal_Control_explicit_count_FT', 'Risk_Internal_Control_implicit_count_FT',
                    'Strategy_explicit_count', 'Strategy_implicit_count', 'Strategy_explicit_count_FT', 'Strategy_implicit_count_FT',
                    'Pricing_Revenue_Management_explicit_count', 'Pricing_Revenue_Management_implicit_count', 'Pricing_Revenue_Management_explicit_count_FT', 'Pricing_Revenue_Management_implicit_count_FT',
                    'Budgeting_Planning_explicit_count', 'Budgeting_Planning_implicit_count',  'Budgeting_Planning_explicit_count_FT', 'Budgeting_Planning_implicit_count_FT']

#Have a look at the resulting counts
count_scores = corpus_df[count_columns]
display(count_scores.describe())

del count_scores

<div class='alert-info'>
Step 3: Create the final MAP dimension measures used in the analysis
</div>

<div class='alert-info'>
Step 3.1: Equally-weighted MAP dimension measures scaled by the total number of sentences
</div>

The first GLLM measure used in the analysis is the equally-weighted MAP measures. It takes the number of total implicit/explicit sentence per dimension per filing and divides it by the total number of sentences. Afterwards the measures are normalized by substracting the minimum and divided by the difference of the maximum minus the minimum of the respective scores.  

In total we creat 3 different kind of scores (repeat step 3.1 & 3.2 with the different settings):
1. MAP scores normalized across the whole sample including Financing/Investment firms (within_industry = False & without_finance = False)
2. MAP scores normalized across the whole sample excluding Financing/Investment firms (within_industry = False & without_finance = True)
3. MAP scores normalized within industry including Financing/Investment firms (within_industry = True & without_finance = False)

In [ ]:
# Load the dataset
corpus_df = pickle.load(open('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl', 'rb'))

# If the normalization should be done within industry, we set the variable 'within_industry' to True. If the normalization should be done across the whole sample, we set it to False.
within_industry = True 

# Specify whether to exclude the Financing/Investment firms from the MAP dimension measure creation (just possible if within_industry=False)
without_finance = False

#If within industry normalization is chosen, we delete all observations with empty industry information and filter for industries with at least 50 observations
if within_industry == True:
    corpus_df = corpus_df[corpus_df['NAICS Sector Name']!=''].copy().reset_index(drop=True)
    corpus_df = corpus_df.iloc[:, :75].copy() # remove potential previous calculations of MAP dimensions measures

# If without finance normalization is chosen, we delete all observations in the Finance and Insurance industry
if without_finance == True:
    corpus_df = corpus_df[corpus_df['NAICS Sector Name']!='Finance and Insurance'].copy().reset_index(drop=True)
    corpus_df = corpus_df.iloc[:, :75].copy() # remove potential previous calculations of MAP dimensions measures

In [ ]:
# Define the list of dimension names
map_dimensions = ['Cost', 
                  'Financing_Investment', 
                  'Operations', 
                  'Performance_Internal_Reporting', 
                  'Risk_Internal_Control', 'Strategy', 
                  'Pricing_Revenue_Management', 
                  'Budgeting_Planning']

# Loop through each dimension and calculate equally-weighted measures
for dim in map_dimensions:
    # Calculate equally-weighted measures for non-FT
    dim_explicit_count = corpus_df[f'{dim}_explicit_count']
    dim_implicit_count = corpus_df[f'{dim}_implicit_count']
    num_entries = corpus_df['num_entries']

    measure_explicit_equally_weighted = dim_explicit_count / num_entries
    measure_implicit_equally_weighted = dim_implicit_count / num_entries

    #Winsorize the measures at the 99th percentile to reduce the influence of extreme outliers
    measure_explicit_equally_weighted = winsorize(measure_explicit_equally_weighted, limits=[0.01, 0.01])
    measure_implicit_equally_weighted = winsorize(measure_implicit_equally_weighted, limits=[0.01, 0.01])

    #Standardize the measures to be between 0 and 1 (if within industry normalization is chosen, consider industry groups)
    if within_industry == True:
        for industry in corpus_df['NAICS Sector Name'].unique():
            industry_filter = corpus_df['NAICS Sector Name'] == industry
            explicit_min = measure_explicit_equally_weighted[industry_filter].min()
            explicit_max = measure_explicit_equally_weighted[industry_filter].max()
            corpus_df.loc[industry_filter, f'{dim}_explicit_equally_standardized'] = (
                (measure_explicit_equally_weighted[industry_filter] - explicit_min) /
                (explicit_max - explicit_min)
            )
            implicit_min = measure_implicit_equally_weighted[industry_filter].min()
            implicit_max = measure_implicit_equally_weighted[industry_filter].max()
            corpus_df.loc[industry_filter, f'{dim}_implicit_equally_standardized'] = (
                (measure_implicit_equally_weighted[industry_filter] - implicit_min) /
                (implicit_max - implicit_min)
            )
    else:
        explicit_min = measure_explicit_equally_weighted.min()
        explicit_max = measure_explicit_equally_weighted.max()
        corpus_df[f'{dim}_explicit_equally_standardized'] = (measure_explicit_equally_weighted - explicit_min) / (explicit_max - explicit_min)
        implicit_min = measure_implicit_equally_weighted.min()
        implicit_max = measure_implicit_equally_weighted.max()
        corpus_df[f'{dim}_implicit_equally_standardized'] = (measure_implicit_equally_weighted - implicit_min) / (implicit_max - implicit_min)

    # Calculate equally-weighted measures for FT
    dim_explicit_count_FT = corpus_df[f'{dim}_explicit_count_FT']
    dim_implicit_count_FT = corpus_df[f'{dim}_implicit_count_FT']

    measure_explicit_equally_weighted_FT = dim_explicit_count_FT / num_entries
    measure_implicit_equally_weighted_FT = dim_implicit_count_FT / num_entries

    #Winsorize the measures at the 99th percentile to reduce the influence of extreme outliers
    measure_explicit_equally_weighted_FT = winsorize(measure_explicit_equally_weighted_FT, limits=[0.01, 0.01])
    measure_implicit_equally_weighted_FT = winsorize(measure_implicit_equally_weighted_FT, limits=[0.01, 0.01])

    #Normalize the measures to be between 0 and 1 (if within industry normalization is chosen, consider industry groups)

    if within_industry == True:
        for industry in corpus_df['NAICS Sector Name'].unique():
            industry_filter = corpus_df['NAICS Sector Name'] == industry
            explicit_min_FT = measure_explicit_equally_weighted_FT[industry_filter].min()
            explicit_max_FT = measure_explicit_equally_weighted_FT[industry_filter].max()
            corpus_df.loc[industry_filter, f'{dim}_explicit_equally_standardized_FT'] = (
                (measure_explicit_equally_weighted_FT[industry_filter] - explicit_min_FT) /
                (explicit_max_FT - explicit_min_FT)
            )
            implicit_min_FT = measure_implicit_equally_weighted_FT[industry_filter].min()
            implicit_max_FT = measure_implicit_equally_weighted_FT[industry_filter].max()
            corpus_df.loc[industry_filter, f'{dim}_implicit_equally_standardized_FT'] = (
                (measure_implicit_equally_weighted_FT[industry_filter] - implicit_min_FT) /
                (implicit_max_FT - implicit_min_FT)
            )
    else:
        explicit_min_FT = measure_explicit_equally_weighted_FT.min()
        explicit_max_FT = measure_explicit_equally_weighted_FT.max()
        corpus_df[f'{dim}_explicit_equally_standardized_FT'] = (measure_explicit_equally_weighted_FT - explicit_min_FT) / (explicit_max_FT - explicit_min_FT)
        implicit_min_FT = measure_implicit_equally_weighted_FT.min()
        implicit_max_FT = measure_implicit_equally_weighted_FT.max()
        corpus_df[f'{dim}_implicit_equally_standardized_FT'] = (measure_implicit_equally_weighted_FT - implicit_min_FT) / (implicit_max_FT - implicit_min_FT)

del dim, dim_explicit_count, dim_implicit_count, dim_explicit_count_FT, dim_implicit_count_FT, num_entries, measure_explicit_equally_weighted, measure_implicit_equally_weighted, measure_explicit_equally_weighted_FT, measure_implicit_equally_weighted_FT, explicit_min, explicit_max, implicit_min, implicit_max, explicit_min_FT, explicit_max_FT, implicit_min_FT, implicit_max_FT

# Save the corpus with the equally-weighted normalized MAP measures included
if within_industry == True and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')


The following code gives a short overview of the equally weighted MAP dimension variables.

In [ ]:
# Dimension columns
equally_columns = ['Cost_explicit_equally_standardized', 'Cost_implicit_equally_standardized', 'Cost_explicit_equally_standardized_FT', 'Cost_implicit_equally_standardized_FT',
           'Financing_Investment_explicit_equally_standardized', 'Financing_Investment_implicit_equally_standardized', 'Financing_Investment_explicit_equally_standardized_FT', 'Financing_Investment_implicit_equally_standardized_FT',
           'Operations_explicit_equally_standardized', 'Operations_implicit_equally_standardized', 'Operations_explicit_equally_standardized_FT', 'Operations_implicit_equally_standardized_FT',
           'Performance_Internal_Reporting_explicit_equally_standardized', 'Performance_Internal_Reporting_implicit_equally_standardized', 'Performance_Internal_Reporting_explicit_equally_standardized_FT', 'Performance_Internal_Reporting_implicit_equally_standardized_FT',
           'Risk_Internal_Control_explicit_equally_standardized', 'Risk_Internal_Control_implicit_equally_standardized', 'Risk_Internal_Control_explicit_equally_standardized_FT', 'Risk_Internal_Control_implicit_equally_standardized_FT',
           'Strategy_explicit_equally_standardized', 'Strategy_implicit_equally_standardized', 'Strategy_explicit_equally_standardized_FT', 'Strategy_implicit_equally_standardized_FT',
           'Pricing_Revenue_Management_explicit_equally_standardized', 'Pricing_Revenue_Management_implicit_equally_standardized', 'Pricing_Revenue_Management_explicit_equally_standardized_FT', 'Pricing_Revenue_Management_implicit_equally_standardized_FT',
           'Budgeting_Planning_explicit_equally_standardized', 'Budgeting_Planning_implicit_equally_standardized',  'Budgeting_Planning_explicit_equally_standardized_FT', 'Budgeting_Planning_implicit_equally_standardized_FT']

#Have a look at the resulting equally-weighted normalized MAP scores
MAP_equally_scores = corpus_df[equally_columns]

display(MAP_equally_scores.describe())

del MAP_equally_scores


<div class='alert-info'>
Step 3.2: CS-weighted MAP dimension measures scaled by the total number of sentences
</div>

With our second measure we want to account for the informational detail (some sentences include more details on MAPs and some less) and, as before, for the length of the document. To do so we weight the occurence of a MAP setence with the respective confidence (CS) and then add up the CS-weigthed number of sentences per dimension. Afterwards, the CS-weigthed score is divided by the length of filing (total number of sentences), and finally normalized on a scale between 0 and 1.

In total we creat 3 different kind of scores:
1. MAP scores normalized across the whole sample including Financing/Investment firms (within_industry = False & without_finance = False)
2. MAP scores normalized across the whole sample excluding Financing/Investment firms (within_industry = False & without_finance = True)
3. MAP scores normalized within industry including Financing/Investment firms (within_industry = True & without_finance = False)

First, we calculate the CS-weighted sum for each filing.

In [ ]:
dimension_map = {
    'Cost': ('Cost_df', 'Cost_FT_df'),
    'Financing_Investment': ('Investment_df', 'Investment_FT_df'),
    'Operations': ('Operations_df', 'Operations_FT_df'),
    'Performance_Internal_Reporting': ('Performance_df', 'Performance_FT_df'),
    'Risk_Internal_Control': ('Risk_df', 'Risk_FT_df'),
    'Strategy': ('Strategy_df', 'Strategy_FT_df'),
    'Pricing_Revenue_Management': ('Pricing_df', 'Pricing_FT_df'),
    'Budgeting_Planning': ('Budget_df', 'Budget_FT_df'),
}

for i in range(len(corpus_df)):
    if i % 50 == 0:
        print('Processing filing ', i, ' of ', len(corpus_df))

    for dim, (col, col_ft) in dimension_map.items():
        # Count number of 'Yes' explicit and implicit sentences for non-FT and weight with LLM_Confidence_Score/100
        df = corpus_df.at[i, col]
        explicit_confidence_scores = df.loc[df['LLM_Explicit_MAP_referral'] == 'Yes', 'LLM_Confidence_Score']
        implicit_confidence_scores = df.loc[df['LLM_Implicit_MAP_referral'] == 'Yes', 'LLM_Confidence_Score']
        
        corpus_df.at[i, f'{dim}_explicit_CS_count'] = explicit_confidence_scores.sum() / 100
        corpus_df.at[i, f'{dim}_implicit_CS_count'] = implicit_confidence_scores.sum() / 100

        # Count number of 'Yes' explicit and implicit sentences for FT and weight with LLM_Confidence_Score
        df_ft = corpus_df.at[i, col_ft]
        explicit_confidence_scores_ft = df_ft.loc[df_ft['LLM_Explicit_MAP_referral'] == 'Yes', 'LLM_Confidence_Score']
        implicit_confidence_scores_ft = df_ft.loc[df_ft['LLM_Implicit_MAP_referral'] == 'Yes', 'LLM_Confidence_Score']
        
        corpus_df.at[i, f'{dim}_explicit_CS_count_FT'] = explicit_confidence_scores_ft.sum() / 100
        corpus_df.at[i, f'{dim}_implicit_CS_count_FT'] = implicit_confidence_scores_ft.sum() / 100
        
del dim, col, col_ft, i, df, df_ft, explicit_confidence_scores, implicit_confidence_scores, explicit_confidence_scores_ft, implicit_confidence_scores_ft, dimension_map

# Save the corpus_df after counting explicit and implicit sentences
if within_industry == True and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')

Second, we divide the CS-weighted measures by the number of sentences in each filing, winsorize the score, and finally normalize.

In [ ]:
# Define the list of dimension names
map_dimensions = ['Cost', 
                  'Financing_Investment', 
                  'Operations', 
                  'Performance_Internal_Reporting', 
                  'Risk_Internal_Control', 'Strategy', 
                  'Pricing_Revenue_Management', 
                  'Budgeting_Planning']

# Loop through each dimension and calculate equally-weighted measures
for dim in map_dimensions:
        # Calculate CS-weighted measures for non-FT      
        
        # Divide by total number of sentences to account for length of filing
        dim_explicit_CS_count = corpus_df[f'{dim}_explicit_CS_count']
        dim_implicit_CS_count = corpus_df[f'{dim}_implicit_CS_count']
        num_entries = corpus_df['num_entries']

        measure_explicit_CS_weighted = dim_explicit_CS_count / num_entries
        measure_implicit_CS_weighted = dim_implicit_CS_count / num_entries
        # Winsorize the measures at the 99th percentile to reduce the influence of extreme outliers
        measure_explicit_CS_weighted = winsorize(measure_explicit_CS_weighted, limits=[0.01, 0.01])
        measure_implicit_CS_weighted = winsorize(measure_implicit_CS_weighted, limits=[0.01, 0.01])
        # Normalize the measures to be between 0 and 1 (if within industry normalization is chosen, consider industry groups)
        if within_industry == True:
            for industry in corpus_df['NAICS Sector Name'].unique():
                industry_filter = corpus_df['NAICS Sector Name'] == industry
                explicit_min = measure_explicit_CS_weighted[industry_filter].min()
                explicit_max = measure_explicit_CS_weighted[industry_filter].max()
                corpus_df.loc[industry_filter, f'{dim}_explicit_CS_standardized'] = (
                    (measure_explicit_CS_weighted[industry_filter] - explicit_min) /
                    (explicit_max - explicit_min)
                )
                implicit_min = measure_implicit_CS_weighted[industry_filter].min()
                implicit_max = measure_implicit_CS_weighted[industry_filter].max()
                corpus_df.loc[industry_filter, f'{dim}_implicit_CS_standardized'] = (
                    (measure_implicit_CS_weighted[industry_filter] - implicit_min) /
                    (implicit_max - implicit_min)
                )
        else:
            explicit_min = measure_explicit_CS_weighted.min()
            explicit_max = measure_explicit_CS_weighted.max()
            corpus_df[f'{dim}_explicit_CS_standardized'] = (measure_explicit_CS_weighted - explicit_min) / (explicit_max - explicit_min)
            implicit_min = measure_implicit_CS_weighted.min()
            implicit_max = measure_implicit_CS_weighted.max()
            corpus_df[f'{dim}_implicit_CS_standardized'] = (measure_implicit_CS_weighted - implicit_min) / (implicit_max - implicit_min)

        # Calculate CS-weighted measures for FT      
        dim_explicit_CS_count_FT = corpus_df[f'{dim}_explicit_CS_count_FT']
        dim_implicit_CS_count_FT = corpus_df[f'{dim}_implicit_CS_count_FT']
        # Divide by total number of sentences to account for length of filing
        measure_explicit_CS_weighted_FT = dim_explicit_CS_count_FT / num_entries
        measure_implicit_CS_weighted_FT = dim_implicit_CS_count_FT / num_entries
        # Winsorize the measures at the 99th percentile to reduce the influence of extreme outliers
        measure_explicit_CS_weighted_FT = winsorize(measure_explicit_CS_weighted_FT, limits=[0.01, 0.01])
        measure_implicit_CS_weighted_FT = winsorize(measure_implicit_CS_weighted_FT, limits=[0.01, 0.01])
        # Normalize the measures to be between 0 and 1 (if within industry normalization is chosen, consider industry groups)
        if within_industry == True:
            for industry in corpus_df['NAICS Sector Name'].unique():
                industry_filter = corpus_df['NAICS Sector Name'] == industry
                explicit_min_FT = measure_explicit_CS_weighted_FT[industry_filter].min()
                explicit_max_FT = measure_explicit_CS_weighted_FT[industry_filter].max()
                corpus_df.loc[industry_filter, f'{dim}_explicit_CS_standardized_FT'] = (
                    (measure_explicit_CS_weighted_FT[industry_filter] - explicit_min_FT) /
                    (explicit_max_FT - explicit_min_FT)
                )
                implicit_min_FT = measure_implicit_CS_weighted_FT[industry_filter].min()
                implicit_max_FT = measure_implicit_CS_weighted_FT[industry_filter].max()
                corpus_df.loc[industry_filter, f'{dim}_implicit_CS_standardized_FT'] = (
                    (measure_implicit_CS_weighted_FT[industry_filter] - implicit_min_FT) /
                    (implicit_max_FT - implicit_min_FT)
                )
        else:             
                explicit_min_FT = measure_explicit_CS_weighted_FT.min()
                explicit_max_FT = measure_explicit_CS_weighted_FT.max()
                corpus_df[f'{dim}_explicit_CS_standardized_FT'] = (measure_explicit_CS_weighted_FT - explicit_min_FT) / (explicit_max_FT - explicit_min_FT)
                implicit_min_FT = measure_implicit_CS_weighted_FT.min()
                implicit_max_FT = measure_implicit_CS_weighted_FT.max()
                corpus_df[f'{dim}_implicit_CS_standardized_FT'] = (measure_implicit_CS_weighted_FT - implicit_min_FT) / (implicit_max_FT - implicit_min_FT)  

del dim, dim_explicit_CS_count, dim_implicit_CS_count, dim_explicit_CS_count_FT, dim_implicit_CS_count_FT, num_entries, measure_explicit_CS_weighted, measure_implicit_CS_weighted, measure_explicit_CS_weighted_FT, measure_implicit_CS_weighted_FT, explicit_min, explicit_max, implicit_min, implicit_max, explicit_min_FT, explicit_max_FT, implicit_min_FT, implicit_max_FT, map_dimensions

# Save the corpus_df after calculating the CS-weighted normalized measures
if within_industry == True and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df.to_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')

In [ ]:
cs_columns = ['Cost_explicit_CS_standardized', 'Cost_implicit_CS_standardized', 'Cost_explicit_CS_standardized_FT', 'Cost_implicit_CS_standardized_FT',
           'Financing_Investment_explicit_CS_standardized', 'Financing_Investment_implicit_CS_standardized', 'Financing_Investment_explicit_CS_standardized_FT', 'Financing_Investment_implicit_CS_standardized_FT',
           'Operations_explicit_CS_standardized', 'Operations_implicit_CS_standardized', 'Operations_explicit_CS_standardized_FT', 'Operations_implicit_CS_standardized_FT',
           'Performance_Internal_Reporting_explicit_CS_standardized', 'Performance_Internal_Reporting_implicit_CS_standardized', 'Performance_Internal_Reporting_explicit_CS_standardized_FT', 'Performance_Internal_Reporting_implicit_CS_standardized_FT',
           'Risk_Internal_Control_explicit_CS_standardized', 'Risk_Internal_Control_implicit_CS_standardized', 'Risk_Internal_Control_explicit_CS_standardized_FT', 'Risk_Internal_Control_implicit_CS_standardized_FT',
           'Strategy_explicit_CS_standardized', 'Strategy_implicit_CS_standardized', 'Strategy_explicit_CS_standardized_FT', 'Strategy_implicit_CS_standardized_FT',
           'Pricing_Revenue_Management_explicit_CS_standardized', 'Pricing_Revenue_Management_implicit_CS_standardized', 'Pricing_Revenue_Management_explicit_CS_standardized_FT', 'Pricing_Revenue_Management_implicit_CS_standardized_FT',
           'Budgeting_Planning_explicit_CS_standardized', 'Budgeting_Planning_implicit_CS_standardized',  'Budgeting_Planning_explicit_CS_standardized_FT', 'Budgeting_Planning_implicit_CS_standardized_FT']

#Have a look at the resulting CS-weighted normalized MAP scores
MAP_CS_scores = corpus_df[cs_columns]

describe(MAP_CS_scores)

del MAP_CS_scores


## Descriptive Analyses of MAP Dimension Scores

In this section, descriptive analyses are performed. First, we will have a look at the raw sentence-count variables per MAP dimension, i.e. evaluate the occurence of the MAP-related sentences in the corpus, including their development of mean over time, and the difference in means across industries. Second, the equally weighted and normalized MAP measures are investigated. Last, we perfom the same analyses for the CS-weigthed and normalized MAP measures. 

<div class='alert-warning'>
Load the corpus dataframe
</div>

In [ ]:
# If the normalization should be done within industry, we set the variable 'within_industry' to True. If the normalization should be done across the whole sample, we set it to False.
within_industry = False

# Specify whether to exclude the Financing/Investment firms from the MAP dimension measure creation (just possible if within_industry=False)
without_finance = False

# Load corpus_df
if within_industry == True and without_finance == False:
    corpus_df = pd.read_pickle('GLLM/Corpus_df_GLLM_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df = pd.read_pickle('GLLM/Corpus_df_GLLM_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df = pd.read_pickle('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')

# Define the columns to be retained in the final DataFrames
count_columns = ['Cost_explicit_count', 'Cost_implicit_count', 'Cost_explicit_count_FT', 'Cost_implicit_count_FT',
                    'Financing_Investment_explicit_count', 'Financing_Investment_implicit_count', 'Financing_Investment_explicit_count_FT', 'Financing_Investment_implicit_count_FT',
                    'Operations_explicit_count', 'Operations_implicit_count', 'Operations_explicit_count_FT', 'Operations_implicit_count_FT',
                    'Performance_Internal_Reporting_explicit_count', 'Performance_Internal_Reporting_implicit_count', 'Performance_Internal_Reporting_explicit_count_FT', 'Performance_Internal_Reporting_implicit_count_FT',
                    'Risk_Internal_Control_explicit_count', 'Risk_Internal_Control_implicit_count', 'Risk_Internal_Control_explicit_count_FT', 'Risk_Internal_Control_implicit_count_FT',
                    'Strategy_explicit_count', 'Strategy_implicit_count', 'Strategy_explicit_count_FT', 'Strategy_implicit_count_FT',
                    'Pricing_Revenue_Management_explicit_count', 'Pricing_Revenue_Management_implicit_count', 'Pricing_Revenue_Management_explicit_count_FT', 'Pricing_Revenue_Management_implicit_count_FT',
                    'Budgeting_Planning_explicit_count', 'Budgeting_Planning_implicit_count',  'Budgeting_Planning_explicit_count_FT', 'Budgeting_Planning_implicit_count_FT']

equally_columns = ['Cost_explicit_equally_standardized', 'Cost_implicit_equally_standardized', 'Cost_explicit_equally_standardized_FT', 'Cost_implicit_equally_standardized_FT',
           'Financing_Investment_explicit_equally_standardized', 'Financing_Investment_implicit_equally_standardized', 'Financing_Investment_explicit_equally_standardized_FT', 'Financing_Investment_implicit_equally_standardized_FT',
           'Operations_explicit_equally_standardized', 'Operations_implicit_equally_standardized', 'Operations_explicit_equally_standardized_FT', 'Operations_implicit_equally_standardized_FT',
           'Performance_Internal_Reporting_explicit_equally_standardized', 'Performance_Internal_Reporting_implicit_equally_standardized', 'Performance_Internal_Reporting_explicit_equally_standardized_FT', 'Performance_Internal_Reporting_implicit_equally_standardized_FT',
           'Risk_Internal_Control_explicit_equally_standardized', 'Risk_Internal_Control_implicit_equally_standardized', 'Risk_Internal_Control_explicit_equally_standardized_FT', 'Risk_Internal_Control_implicit_equally_standardized_FT',
           'Strategy_explicit_equally_standardized', 'Strategy_implicit_equally_standardized', 'Strategy_explicit_equally_standardized_FT', 'Strategy_implicit_equally_standardized_FT',
           'Pricing_Revenue_Management_explicit_equally_standardized', 'Pricing_Revenue_Management_implicit_equally_standardized', 'Pricing_Revenue_Management_explicit_equally_standardized_FT', 'Pricing_Revenue_Management_implicit_equally_standardized_FT',
           'Budgeting_Planning_explicit_equally_standardized', 'Budgeting_Planning_implicit_equally_standardized',  'Budgeting_Planning_explicit_equally_standardized_FT', 'Budgeting_Planning_implicit_equally_standardized_FT']

cs_columns = ['Cost_explicit_CS_standardized', 'Cost_implicit_CS_standardized', 'Cost_explicit_CS_standardized_FT', 'Cost_implicit_CS_standardized_FT',
           'Financing_Investment_explicit_CS_standardized', 'Financing_Investment_implicit_CS_standardized', 'Financing_Investment_explicit_CS_standardized_FT', 'Financing_Investment_implicit_CS_standardized_FT',
           'Operations_explicit_CS_standardized', 'Operations_implicit_CS_standardized', 'Operations_explicit_CS_standardized_FT', 'Operations_implicit_CS_standardized_FT',
           'Performance_Internal_Reporting_explicit_CS_standardized', 'Performance_Internal_Reporting_implicit_CS_standardized', 'Performance_Internal_Reporting_explicit_CS_standardized_FT', 'Performance_Internal_Reporting_implicit_CS_standardized_FT',
           'Risk_Internal_Control_explicit_CS_standardized', 'Risk_Internal_Control_implicit_CS_standardized', 'Risk_Internal_Control_explicit_CS_standardized_FT', 'Risk_Internal_Control_implicit_CS_standardized_FT',
           'Strategy_explicit_CS_standardized', 'Strategy_implicit_CS_standardized', 'Strategy_explicit_CS_standardized_FT', 'Strategy_implicit_CS_standardized_FT',
           'Pricing_Revenue_Management_explicit_CS_standardized', 'Pricing_Revenue_Management_implicit_CS_standardized', 'Pricing_Revenue_Management_explicit_CS_standardized_FT', 'Pricing_Revenue_Management_implicit_CS_standardized_FT',
           'Budgeting_Planning_explicit_CS_standardized', 'Budgeting_Planning_implicit_CS_standardized',  'Budgeting_Planning_explicit_CS_standardized_FT', 'Budgeting_Planning_implicit_CS_standardized_FT']


analysis_df = corpus_df[['filing_key', 'filing_year', 'NAICS Sector Name', 'NAICS Subsector Name'] + count_columns + equally_columns + cs_columns].copy()

del corpus_df

# Define the mapping of column names to more readable labels
col_dim_map = {
    'Cost': 'Cost',
    'Financing_Investment': 'Financing/Investment',
    'Operations': 'Operations',
    'Performance_Internal_Reporting': 'Performance/Internal Reporting',
    'Risk_Internal_Control': 'Risk/Internal Control',
    'Strategy': 'Strategy',
    'Pricing_Revenue_Management': 'Pricing/Revenue Management',
    'Budgeting_Planning': 'Budgeting/Planning'
}

<div class='alert-warning'>
Set output path and size for figures/plots
</div>

In [ ]:
#Directory to save plots
output_dir_plots_raw = 'Analyses_outputs/Plots/MAP_Dimension_Scores/GLLM/Raw count'
if not os.path.exists(output_dir_plots_raw):
    os.makedirs(output_dir_plots_raw)

output_dir_plots_equally = 'Analyses_outputs/Plots/MAP_Dimension_Scores/GLLM/Equally_weighted_normalized'
if not os.path.exists(output_dir_plots_equally):
    os.makedirs(output_dir_plots_equally)

output_dir_plots_cs =  'Analyses_outputs/Plots/MAP_Dimension_Scores/GLLM/CS_weighted_normalized'
if not os.path.exists(output_dir_plots_cs):
    os.makedirs(output_dir_plots_cs)

#Define the figure size for the output
plotnine.options.figure_size = (12,6)

<div class='alert-info'>
Step 1: Raw Sentence Count Analyses
</div>

In the following we use the previously created variables '..._count' to perform the analyses. We will have a look at a summary statistic, a frequency plot, the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the raw sentence count per dimension.

In [ ]:
# Print descriptive statistics for the count columns
display(analysis_df[count_columns].describe())

Second, we have a look at the total frequency of MAP-related sentences per dimension and save it as frequency plot.


In [ ]:
#We create a figure that displays the total sum for each raw sentence count MAP measure

#Frist, we sum up the total frequency of tokens per dimension
total_frequencies = analysis_df[count_columns].sum().reset_index()

#Second, we rename the columns and the dimensions
total_frequencies.columns = ['Dimension','Frequency']

#Third, we save the total frequency (sum per dimension) as integer value 
total_frequencies['Frequency'] = total_frequencies['Frequency'].astype(int)

#Fourth, create a new column in the DataFrame with comma formatting
total_frequencies['Formatted_Frequency'] = total_frequencies['Frequency'].apply(lambda x: f'{x:,}')

#Fifth, create the plots
map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

for map_referral, ft_type in map_type:
    if ft_type == 'Non-FT':
        plot_type = ''
    else:
        plot_type = '_FT'
        
    # Filter the total_frequencies DataFrame for the current map_referral and ft_type
    if ft_type == 'Non-FT':
        filtered_frequencies = total_frequencies[total_frequencies['Dimension'].str.contains(map_referral.lower()) & ~total_frequencies['Dimension'].str.contains('FT')].copy()
    else:
        filtered_frequencies = total_frequencies[total_frequencies['Dimension'].str.contains(map_referral.lower()) & total_frequencies['Dimension'].str.contains(ft_type)].copy()

    filtered_frequencies['Dimension'] = filtered_frequencies['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)
    
    # Create the y-axis label based on the current map_referral and ft_type
    y_axis_label = f'Total {map_referral} Sentence Frequency ({ft_type})'

    maximum_frequency = filtered_frequencies['Frequency'].max()

    plot = (plotnine.ggplot(filtered_frequencies, plotnine.aes(
        x='reorder(Dimension, Frequency)', 
        y='Frequency',
        label='Formatted_Frequency',  # Apply comma formatting 
        fill='Dimension'))
        # We define that the length of the bar equals the total frequency of the respective dimension
        + plotnine.geom_bar(stat='identity') 
        # We swap the x and y-axis 
        + plotnine.coord_flip()
        # We change the x-axis label
        + plotnine.xlab('MAP Dimension')
        # ... and the y-axis label
        + plotnine.ylab(y_axis_label)
        # We add the number of the respective frequency at the right end of the bar 
        + plotnine.geom_text(ha='left', va='center', color='black', size=12, fontweight='bold')
        # Format y-axis with commas as thousand separators and adjust the limits such that the labels are displayed properly
        + plotnine.scale_y_continuous(labels=comma_format(), limits=(0, maximum_frequency * 1.1))
        # Change the appearance (white background, bold title, etc..)
        + plotnine.theme(
            legend_position='none',
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
        
    )

    #Last, save the plot to the ouput directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'{map_referral}_Sentence_Frequency_Plot_Raw_MAP_Dimensions_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_raw, f'{map_referral}_Sentence_Frequency_Plot_Raw_MAP_Dimensions_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'{map_referral}_Sentence_Frequency_Plot_Raw_MAP_Dimensions{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del total_frequencies, filtered_frequencies, plot, map_type, plot_type, y_axis_label, maximum_frequency


Third, we investigate the development of the means over time.

In [ ]:
#3. Create a figure that display the development of the sentence frequency for each dimension (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_year_avg_df = analysis_df.groupby('filing_year')[count_columns].mean().reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Second, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    count_columns_tmp = count_columns.copy()
    if ft_type == 'Non-FT':
        count_columns_tmp = [col for col in count_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        count_columns_tmp = [col for col in count_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimensions_year_avg_df_tmp = MAP_dimensions_year_avg_df[['filing_year'] + count_columns_tmp].copy()

    #We transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
    MAP_dimensions_year_avg_df_long = MAP_dimensions_year_avg_df_tmp.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

    #We rename the columns of the dimensions
    MAP_dimensions_year_avg_df_long['Dimension'] = MAP_dimensions_year_avg_df_long['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #We define the range of years in the corpus (needed to set axis limits in the plot)
    years = MAP_dimensions_year_avg_df_long['filing_year'].unique()
    year_min = years.min()
    year_max = years.max()

    #Create one plot with all variables
    y_axis_label = f'Average {map_referral} Sentence Frequency ({ft_type})'

    plot = (plotnine.ggplot(MAP_dimensions_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
        + plotnine.geom_line(size=1.5) 
        + plotnine.labs(
            x='Filing Year',
            y=y_axis_label)
        + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
        + plotnine.theme(
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Save the plot to the output directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'Development_{map_referral}_Frequency_Raw_MAP_Dimensions_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_raw, f'Development_{map_referral}_Frequency_Raw_MAP_Dimensions_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'Development_{map_referral}_Frequency_Raw_MAP_Dimensions{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimensions_year_avg_df_tmp, MAP_dimensions_year_avg_df_long, years, plot, map_type, plot_type, y_axis_label, count_columns_tmp

Fourth, we analyse the means across NAICS industries.

In [ ]:
#4. Create plots that show the average sentence frequency per industry and dimension (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis and create a subset of the analysis dataframe with firms that have a valid industry classification
columns = ['filing_year', 'NAICS Sector Name'] + count_columns

analysis_df_ind = analysis_df[analysis_df['NAICS Sector Name']!=''].copy()

#Second, transform the dataframe to a long format (1 filing has 8 rows - one for each dimension)
MAP_dimension_industry_long_df = analysis_df_ind[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Frequency')

#Third, calculate the sentence frequency mean for each group (Industry,Dimension)
MAP_dimension_industry_long_avg_df = MAP_dimension_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Frequency': 'mean'}).reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Third, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    count_columns_tmp = count_columns.copy()
    if ft_type == 'Non-FT':
        count_columns_tmp = [col for col in count_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        count_columns_tmp = [col for col in count_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimension_industry_long_avg_df_tmp = MAP_dimension_industry_long_avg_df[MAP_dimension_industry_long_avg_df['Dimension'].isin(count_columns_tmp)].copy()

    #We rename the columns of the dimensions (redo)
    MAP_dimension_industry_long_avg_df_tmp['Dimension'] = MAP_dimension_industry_long_avg_df_tmp['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #Create one plot with all variables
    y_axis_label = f'Average {map_referral} Sentence Frequency ({ft_type})'
    plot = (
        plotnine.ggplot(MAP_dimension_industry_long_avg_df_tmp, plotnine.aes(x='NAICS Sector Name', y='Frequency', color='Dimension', group='Dimension'))
        + plotnine.geom_point(size=4)
        + plotnine.geom_line(size=1)
        + plotnine.xlab('Industry')
        + plotnine.ylab(y_axis_label)
        + plotnine.theme(
            axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Sixth, save the plot to the ouput directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'Raw_MAP_Dimensions_{map_referral}_Frequency_per_Industry_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_raw, f'Raw_MAP_Dimensions_{map_referral}_Frequency_per_Industry_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_raw, f'Raw_MAP_Dimensions_{map_referral}_Frequency_per_Industry{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimension_industry_long_df, MAP_dimension_industry_long_avg_df_tmp, plot, map_type, plot_type, y_axis_label, count_columns_tmp, columns


<div class='alert-info'>
Step 2: Equally Weighted and normalized MAP Measures Analyses 
</div>

Now, we have a look at the equally weighted and normalized MAP measures, i.e. each measures is a equally weighted sum of the respective MAP sentences, divided by the total number of sentences in the filing, and normalized (such that they are between 0 and 1). We use the previously created variables '..._equally_standardized' to perform the analyses. As for the raw sentence count, we look at a summary statistic, the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the equally weighted and normalized MAP measure per dimension.

In [ ]:
analysis_df[equally_columns].describe()

Second, we investigate the development of the means over time.

In [ ]:
#3.1 Create a figure that display the development of the equally weighted and normalized MAP measure for each dimension (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_year_avg_df = analysis_df.groupby('filing_year')[equally_columns].mean().reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Second, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    equally_columns_tmp = equally_columns.copy()
    if ft_type == 'Non-FT':
        equally_columns_tmp = [col for col in equally_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        equally_columns_tmp = [col for col in equally_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimensions_year_avg_df_tmp = MAP_dimensions_year_avg_df[['filing_year'] + equally_columns_tmp].copy()

    #We transform the dataframe to a long format (1 filing has 8 rows - one for each dimension)
    MAP_dimensions_year_avg_df_long = MAP_dimensions_year_avg_df_tmp.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

    #We rename the columns of the dimensions
    MAP_dimensions_year_avg_df_long['Dimension'] = MAP_dimensions_year_avg_df_long['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #We define the range of years in the corpus (needed to set axis limits in the plot)
    years = MAP_dimensions_year_avg_df_long['filing_year'].unique()
    year_min = years.min()
    year_max = years.max()
    map_max = np.round(MAP_dimensions_year_avg_df_long['Average'].max(), 1)
    #Create one plot with all variables
    y_axis_label = f'Average Equally Weighted {map_referral} MAP Measures ({ft_type})'

    plot = (plotnine.ggplot(MAP_dimensions_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
        + plotnine.geom_line(size=1.5) 
        + plotnine.labs(
            x='Filing Year',
            y=y_axis_label)
        + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
        + plotnine.scale_y_continuous(breaks=np.arange(0, map_max + 0.15, 0.1), limits=(0, map_max+0.1))
        + plotnine.theme(
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Save the plot to the output directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_equally, f'Development_Equally_Weighted_{map_referral}_MAP_Measures_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_equally, f'Development_Equally_Weighted_{map_referral}_MAP_Measures_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_equally, f'Development_Equally_Weighted_{map_referral}_MAP_Measures{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimensions_year_avg_df_tmp, MAP_dimensions_year_avg_df_long, years, plot, map_type, plot_type, y_axis_label, equally_columns_tmp

Third, we analyse the means across industries.

In [ ]:
#3. Create plots that show the equally weighted and normalized MAP measures per industry and dimension (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis and create a subset of the analysis dataframe with firms that have a valid industry classification
columns = ['filing_year', 'NAICS Sector Name'] + equally_columns

analysis_df_ind = analysis_df[analysis_df['NAICS Sector Name']!=''].copy()

#Second, transform the dataframe to a long format (1 filing has 8 rows - one for each dimension)
MAP_dimension_industry_long_df = analysis_df_ind[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Frequency')

#Third, calculate the average equally weighted MAP measure for each group (Industry,Dimension)
MAP_dimension_industry_long_avg_df = MAP_dimension_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Frequency': 'mean'}).reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Fourth, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    equally_columns_tmp = equally_columns.copy()
    if ft_type == 'Non-FT':
        equally_columns_tmp = [col for col in equally_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        equally_columns_tmp = [col for col in equally_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimension_industry_long_avg_df_tmp = MAP_dimension_industry_long_avg_df[MAP_dimension_industry_long_avg_df['Dimension'].isin(equally_columns_tmp)].copy()

    #We rename the columns of the dimensions 
    MAP_dimension_industry_long_avg_df_tmp['Dimension'] = MAP_dimension_industry_long_avg_df_tmp['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #Create one plot with all variables
    y_axis_label = f'Average Equally Weighted {map_referral} MAP Measures ({ft_type})'
    plot = (
        plotnine.ggplot(MAP_dimension_industry_long_avg_df_tmp, plotnine.aes(x='NAICS Sector Name', y='Frequency', color='Dimension', group='Dimension'))
        + plotnine.geom_point(size=4)
        + plotnine.geom_line(size=1)
        + plotnine.xlab('Industry')
        + plotnine.ylab(y_axis_label)
        + plotnine.scale_y_continuous(breaks=np.arange(0, MAP_dimension_industry_long_avg_df_tmp['Frequency'].max() + 0.1, 0.1), limits=(0, MAP_dimension_industry_long_avg_df_tmp['Frequency'].max()+0.1))
        + plotnine.theme(
            axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Sixth, save the plot to the ouput directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_equally, f'Equally_Weighted_{map_referral}_MAP_Measures_per_Industry_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_equally, f'Equally_Weighted_{map_referral}_MAP_Measures_per_Industry_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_equally, f'Equally_Weighted_{map_referral}_MAP_Measures_per_Industry{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimension_industry_long_df, MAP_dimension_industry_long_avg_df_tmp, plot, map_type, plot_type, y_axis_label, equally_columns_tmp, columns

<div class='alert-info'>
Step 3: Confidence Score (CS) Weighted and normalized MAP Measures Analyses 
</div>

Now, we have a look at the CS weighted and normalized MAP measures, i.e. each measures is a CS weighted sum of the respective sentences, divided by the total number of sentences in the filing, and normalized (such that they are between 0 and 1). We use the previously created variables '..._cs_standardized' to perform the analyses. As mentioned before we look at a summary statistic, the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the CS weighted and normalized MAP measure per dimension.

In [ ]:
display(analysis_df[cs_columns].describe())

Second, we investigate the development of the means over time.

In [ ]:
#2. Create a figure that display the development of the CS weighted and normalized MAP measure for each dimension (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_year_avg_df = analysis_df.groupby('filing_year')[cs_columns].mean().reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Second, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    cs_columns_tmp = cs_columns.copy()
    if ft_type == 'Non-FT':
        cs_columns_tmp = [col for col in cs_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        cs_columns_tmp = [col for col in cs_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimensions_year_avg_df_tmp = MAP_dimensions_year_avg_df[['filing_year'] + cs_columns_tmp].copy()

    #We transform the dataframe to a long format (1 filing has 8 rows - one for each dimension)
    MAP_dimensions_year_avg_df_long = MAP_dimensions_year_avg_df_tmp.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

    #We rename the columns of the dimensions
    MAP_dimensions_year_avg_df_long['Dimension'] = MAP_dimensions_year_avg_df_long['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #We define the range of years in the corpus (needed to set axis limits in the plot)
    years = MAP_dimensions_year_avg_df_long['filing_year'].unique()
    year_min = years.min()
    year_max = years.max()
    map_max = np.round(MAP_dimensions_year_avg_df_long['Average'].max(), 1)

    #Create one plot with all variables
    y_axis_label = f'Average CS-Weighted MAP Measures'

    plot = (plotnine.ggplot(MAP_dimensions_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
        + plotnine.geom_line(size=1.5) 
        + plotnine.labs(
            x='Filing Year',
            y=y_axis_label)
        + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
        + plotnine.scale_y_continuous(breaks=np.arange(0, map_max + 0.15, 0.1), limits=(0, map_max+0.1))
        + plotnine.theme(
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_x=plotnine.element_text(size=14, weight='bold'),  
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Save the plot to the output directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_cs, f'Development_CS_Weighted_{map_referral}_MAP_Measures_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_cs, f'Development_CS_Weighted_{map_referral}_MAP_Measures_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_cs, f'Development_CS_Weighted_{map_referral}_MAP_Measures{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimensions_year_avg_df_tmp, MAP_dimensions_year_avg_df_long, years, plot, map_type, plot_type, y_axis_label, cs_columns_tmp

Third, we analyse the means across industries.

In [ ]:
#3. Create plots that show the CS weighted and normalized MAP measures per industry and dimension (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis
columns = ['filing_year', 'NAICS Sector Name'] + cs_columns

#Second, create a subset of the analysis dataframe with firms that have a valid industry classification
analysis_df_ind = analysis_df[analysis_df['NAICS Sector Name']!=''].copy()

#Second, transform the dataframe to a long format (1 filing has 8 rows - one for each dimension)
MAP_dimension_industry_long_df = analysis_df_ind[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Frequency')

#Third, calculate the average CS weighted MAP measure for each group (Industry,Dimension)
MAP_dimension_industry_long_avg_df = MAP_dimension_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Frequency': 'mean'}).reset_index()

map_type =[('Explicit', 'Non-FT'), ('Implicit', 'Non-FT'), ('Explicit', 'FT'), ('Implicit', 'FT')]

#Fourth, we loop over the different map types to create the respective plots
for map_referral, ft_type in map_type:
    cs_columns_tmp = cs_columns.copy()
    if ft_type == 'Non-FT':
        cs_columns_tmp = [col for col in cs_columns_tmp if 'FT' not in col and map_referral.lower() in col]
        plot_type = ''
    else:
        cs_columns_tmp = [col for col in cs_columns_tmp if 'FT' in col and map_referral.lower() in col]
        plot_type = '_FT'
    
    MAP_dimension_industry_long_avg_df_tmp = MAP_dimension_industry_long_avg_df[MAP_dimension_industry_long_avg_df['Dimension'].isin(cs_columns_tmp)].copy()

    #We rename the columns of the dimensions 
    MAP_dimension_industry_long_avg_df_tmp['Dimension'] = MAP_dimension_industry_long_avg_df_tmp['Dimension'].apply(lambda x: x.split('_explicit')[0].split('_implicit')[0]).replace(col_dim_map)

    #Create one plot with all variables
    y_axis_label = f'Average CS-Weighted MAP Measure'
    plot = (
        plotnine.ggplot(MAP_dimension_industry_long_avg_df_tmp, plotnine.aes(x='NAICS Sector Name', y='Frequency', color='Dimension', group='Dimension'))
        + plotnine.geom_point(size=4)
        + plotnine.geom_line(size=1)
        + plotnine.xlab('Industry')
        + plotnine.ylab(y_axis_label)
        + plotnine.scale_y_continuous(breaks=np.arange(0, MAP_dimension_industry_long_avg_df_tmp['Frequency'].max() + 0.1, 0.1), limits=(0, MAP_dimension_industry_long_avg_df_tmp['Frequency'].max()+0.1))
        + plotnine.theme(
            axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),
            panel_background= plotnine.element_rect(fill='white'),
            plot_title=plotnine.element_text(size=22, weight='bold'),  
            axis_title_x=plotnine.element_text(size=18, weight='bold'),  
            axis_title_y=plotnine.element_text(size=18, weight='bold'),  
            legend_title=plotnine.element_text(size=18),
            legend_text= plotnine.element_text(size=16),
            axis_text_y=plotnine.element_text(size=14, weight='bold')
        )
    )

    #Sixth, save the plot to the ouput directory 
    if within_industry == True and without_finance == False:
        plot.save(os.path.join(output_dir_plots_cs, f'CS_Weighted_{map_referral}_MAP_Measures_per_Industry_within_industry_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == True:
        plot.save(os.path.join(output_dir_plots_cs, f'CS_Weighted_{map_referral}_MAP_Measures_per_Industry_without_finance_normalized{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    elif within_industry == False and without_finance == False:
        plot.save(os.path.join(output_dir_plots_cs, f'CS_Weighted_{map_referral}_MAP_Measures_per_Industry{plot_type}.png'), width=19.2, height=9.67, dpi=300)
    else:
        print('Please check your settings for within_industry and without_finance')

del MAP_dimension_industry_long_df, MAP_dimension_industry_long_avg_df_tmp, plot, map_type, plot_type, y_axis_label, cs_columns_tmp, columns